# Day 2 — Find XRD peaks automatically and calculate crystallite size (Scherrer equation)
**Python for Chemists** series · Muhammad Amjad

Yesterday (Day 1) we plotted an XRD pattern. Today Python will:
1. find every peak by itself,
2. measure each peak's width (FWHM),
3. calculate crystallite size with the **Scherrer equation**:

$$D = \frac{K\lambda}{\beta \cos\theta}$$

- *K* = 0.9 (shape factor) · *λ* = 1.5406 Å (Cu Kα) · *β* = FWHM in **radians** · *θ* = half of 2θ

Sample data: `MnFe2O4_xrd.csv` (a spinel ferrite pattern). Replace it with your own two-column file (2θ, intensity).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

## Step 1 — Read the data

In [ ]:
# Step 1: read the XRD data (2-theta, intensity)
df = pd.read_csv("MnFe2O4_xrd.csv")
x, y = df["two_theta"].values, df["intensity"].values

## Step 2 — Find the peaks
`prominence` = how much a peak must stand above its surroundings. Raise it if noise is detected as peaks; lower it if small peaks are missed.

In [ ]:
# Step 2: let Python find the peaks for you
peaks, _ = find_peaks(y, prominence=80, distance=50)

## Step 3 — Fit each peak and apply the Scherrer equation
A Gaussian is fitted to a ±1° window around each peak. Its width gives FWHM; FWHM is converted to radians before use.

In [ ]:
# Step 3: fit a Gaussian to each peak to get its exact centre and width (FWHM)
def gauss(x, A, c, s, b):
    return A*np.exp(-0.5*((x-c)/s)**2) + b

wavelength = 1.5406   # Cu K-alpha, Angstrom
K = 0.9               # shape factor
results = []
for p in peaks:
    win = (x > x[p]-1.0) & (x < x[p]+1.0)
    guess = [y[p]-y[win].min(), x[p], 0.15, y[win].min()]
    (A, c, s, b), _ = curve_fit(gauss, x[win], y[win], p0=guess)
    fwhm = 2.3548*abs(s)                         # degrees
    beta = np.radians(fwhm)                      # radians
    theta = np.radians(c/2)
    D = K*wavelength/(beta*np.cos(theta))/10     # nm  (Scherrer equation)
    results.append([round(c,2), round(fwhm,3), round(D,1)])

table = pd.DataFrame(results, columns=["2theta (deg)", "FWHM (deg)", "Crystallite size D (nm)"])
print(table)
print("Average crystallite size:", round(table["Crystallite size D (nm)"].mean(),1), "nm")

## Step 4 — Plot

In [ ]:
# Step 4: plot the pattern with the detected peaks marked
labels = ["(220)", "(311)", "(400)", "(422)", "(511)", "(440)"]
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=200)
ax.plot(x, y, color="#1f3b73", lw=1)
ax.plot(x[peaks], y[peaks], "v", color="#d1495b", ms=7, label="Peaks found by Python")
for p, lab, D in zip(peaks, labels, table["Crystallite size D (nm)"]):
    ax.annotate(f"{lab}\n{D} nm", (x[p], y[p]), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8)
ax.set_xlabel("2θ (degree)")
ax.set_ylabel("Intensity (a.u.)")
ax.set_title("MnFe$_2$O$_4$ — automatic peak finding + Scherrer crystallite size")
ax.set_ylim(top=y.max()*1.25)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("day02_scherrer.png")
plt.show()

## Important notes for real data
- **Instrumental broadening:** your diffractometer also widens peaks. For accurate sizes, measure a standard (e.g. Si or LaB₆) and use β = √(β_sample² − β_instrument²).
- Scherrer ignores **strain** broadening. For size + strain, use a Williamson–Hall plot (coming in a later day).
- Scherrer is reliable roughly for crystallites below ~100 nm.
- Remove Kα2 or use a pseudo-Voigt fit if your peaks are asymmetric.